In [117]:
################ Import necessary libraries

%pip install pyblp
%pip install statsmodels
%pip install linearmodels
%pip install stargazer

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
from linearmodels import IV2SLS
from stargazer.stargazer import Stargazer
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import matplotlib.pyplot as plt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [118]:
################ Import data
df = pd.read_csv('model_ready.csv')
supply_df = pd.read_csv('supply_ready.csv')
df = df.reset_index(drop=True)

# 将 supply_df 中的 road_fuel_IV 合并到 df
# 假设合并键为 province 和 year

df = df.merge(
    supply_df[['province', 'year', 'road_fuel_IV', 'num_models_in_market']],
    on=['province', 'year'],
    how='left'
)

# 计算 log(sjm), log(s0m), log(sj/g)
df['log_sjm'] = np.log(df['shares'])
df['log_s0m'] = np.log(1 - df.groupby('market_ids')['shares'].transform('sum'))
df['log_sj_g'] = np.log(df['shares'] / df.groupby(['market_ids', 'nesting_ids'])['shares'].transform('sum'))
df['log_charging_stock'] = np.log(df['charging_stations_stock'])

# Set range, log_charging_stock, log_charging_IV, and battery_capacity to 0 for non-EVs
df.loc[df['is_electric'] == 0, ['range', 'battery_capacity']] = 0



In [119]:
# Define list of IVs
iv_list = [
    'cost_shifter', 
    'product_set_size',
    'euclidean_range',
    'local_range',
    'euclidean_battery',
    'local_battery',
    'euclidean_power',
    'num_models_in_market',
    'road_fuel_IV',
    'local_power',
]

iv_str = ' + '.join(iv_list)

In [120]:
################ 2SLS model using custom instruments

# First stage for charging station: regress log_charging_stock on instrument variables
X = sm.add_constant(df['log_charging_IV'])
y = df['log_charging_stock']
first_stage = sm.OLS(y, X).fit()
df['log_charging_stock_hat'] = first_stage.predict(X)

# Define the formula for the IV2SLS model
formula = f'''
(log_sjm - log_s0m) ~ 0 + is_electric*log_charging_stock_hat + range + power + battery_capacity
    + [net_prices + log_sj_g ~ {iv_str}]
'''
# CAN HAVE HORSEPOWER = POWER / MASS #

iv_model = IV2SLS.from_formula(formula, data=df).fit(cov_type="clustered", clusters=df['market_ids'])

print(iv_model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                log_sjm   R-squared:                      0.9823
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9823
No. Observations:               33545   F-statistic:                 7.239e+04
Date:                Mon, Aug 04 2025   P-value (F-stat)                0.0000
Time:                        16:59:10   Distribution:                  chi2(8)
Cov. Estimator:             clustered                                         
                                                                              
                                         Parameter Estimates                                          
                                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------------------------
is_electric                           -15.658     0.6756   

In [121]:
# Output demand results as LaTeX table using Stargazer, only show selected coefficients
demand_stargazer = Stargazer([iv_model])
demand_stargazer.covariate_order([
    'net_prices',
    'log_sj_g',
    'is_electric:log_charging_stock_hat',
    'is_electric',
    'log_charging_stock_hat',
    'range',
    'power',
    'battery_capacity'
])
demand_stargazer.rename_covariates({
    'net_prices': 'price',
    'log_sj_g': 'nesting coefficient',
    'is_electric': r'$EV_j$',
    'is_electric:log_charging_stock_hat': r'$(\log(N_m) \times EV_j)$',
    'log_charging_stock_hat': r'$\log(N_m)$',
    'battery_capacity': 'battery capacity'
})
latex_demand = demand_stargazer.render_latex()

# Optionally, save to file
with open('GraphsTables/demand_model_result.tex', 'w', encoding='utf-8') as f:
    f.write(latex_demand)

In [122]:
################ Supply side model

# Create a time trend variable
supply_df['time_trend'] = supply_df['year'].astype(int) - supply_df['year'].astype(int).min() + 1

# Add time_trend
formula = 'log(charging_stations_stock) ~ [log(EV_stock) ~ road_fuel_IV + num_models_in_market + sales_weighted_avg_range] + sub_fix + sub_ope + C(province) + time_trend'

supply_model = IV2SLS.from_formula(formula, data=supply_df).fit()
print(supply_model.summary)

                               IV-2SLS Estimation Summary                               
Dep. Variable:     log(charging_stations_stock)   R-squared:                      0.9734
Estimator:                              IV-2SLS   Adj. R-squared:                 0.9659
No. Observations:                           155   F-statistic:                 9.753e+05
Date:                          Mon, Aug 04 2025   P-value (F-stat)                0.0000
Time:                                  16:59:11   Distribution:                 chi2(35)
Cov. Estimator:                          robust                                         
                                                                                        
                                      Parameter Estimates                                      
                             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------------


In [123]:
# Export model summary as LaTeX
supply_stargazer = Stargazer([supply_model])
supply_stargazer.covariate_order(['sub_fix', 'sub_ope', 'log(EV_stock)', 'time_trend'])
supply_stargazer.rename_covariates({
    'sub_fix': 'fixed subsidy',
    'sub_ope': 'operating subsidy',
    'log(EV_stock)': r'$\lambda_1$',
    'time_trend': 'time trend'
})
supply_stargazer.significant_digits(5)
latex_supply = supply_stargazer.render_latex()

# Optionally, save to file
with open('GraphsTables/supply_model_result.tex', 'w', encoding='utf-8') as f:
    f.write(latex_supply)

In [124]:
# Save demand model results
with open('demand_model_results.pkl', 'wb') as f:
    pickle.dump(iv_model, f)

# Save supply model results
with open('charging_station_model_results.pkl', 'wb') as f:
    pickle.dump(supply_model, f)

In [125]:
# Export model data for counterfactual analysis
df.to_csv('demand_counterfactual.csv', index=False)
supply_df.to_csv('supply_counterfactual.csv', index=False)

In [126]:
# Demand model results
with open('demand_model_results.pkl', 'rb') as f:
     demand_param = pickle.load(f)

# Charging station model results
with open('charging_station_model_results.pkl', 'rb') as f:
     charging_param = pickle.load(f)

In [127]:
def compute_nested_logit_derivatives(
    df,
    demand_par,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model_id',
    nest_col='nesting_ids',
    within_share_col='within_nest_shares'
):
    """
    Compute derivatives of market shares with respect to prices for each market using equations 12, 14, 15.
    Returns derivatives ∂s_j/∂p_k, not elasticities.
    Returns:
        dict: {market_id: derivative_matrix (DataFrame, index/columns=model_id)}
    """
    alpha = demand_par.params['net_prices']
    sigma = demand_par.params['log_sj_g']
    derivative_matrices = {}

    # Use existing within-nest shares from the dataset
    # No need to recalculate since they're already in within_nest_shares column

    unique_markets = df[market_col].unique()
    for market_id in unique_markets:
        df_mkt = df[df[market_col] == market_id].copy()
        n = len(df_mkt)
        derivatives = np.zeros((n, n))
        shares = df_mkt['shares'].values
        within_nest_shares = df_mkt[within_share_col].values
        nests = df_mkt[nest_col].values
        model_ids = df_mkt[model_col].values

        # Use existing nest_shares for group shares
        group_shares = df_mkt['nest_shares'].values

        for j in range(n):
            s_j = shares[j]
            s_jg = within_nest_shares[j]
            s_g = group_shares[j]
            for k in range(n):
                s_k = shares[k]
                if j == k:
                    # Own-price derivative (Equation 12)
                    deriv = (1/(1-sigma)) * s_j * (1 - sigma*s_jg - (1-sigma)*s_j)
                elif nests[j] == nests[k]:
                    # Cross-price derivative within the same group (Equation 14)
                    deriv = -s_j * s_k * (1 + (sigma/(1-sigma)) * (1/s_g))
                else:
                    # Cross-price derivative across groups (Equation 15)
                    deriv = -s_j * s_k
                # Apply price coefficient to get derivative ∂s_j/∂p_k
                derivatives[j, k] = alpha * deriv

        derivative_df = pd.DataFrame(derivatives, index=model_ids, columns=model_ids)
        derivative_matrices[market_id] = derivative_df

    return derivative_matrices

In [128]:
# Compute elasticities with network effects
derivative_matrices = compute_nested_logit_derivatives(
    df,
    demand_param,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model',
    nest_col='nesting_ids',
    within_share_col='within_nest_shares'
)

In [129]:
# Compute charging station semi-elasticities (γ_j)
def compute_gamma_dict(beta_N, df, nest_col='nesting_ids', market_col='market_ids', share_col='shares', within_share_col='within_nest_shares', sigma=None):
    """
    Compute charging station semi-elasticities (γ_j) for all products, returned as a dict by market.
    Uses existing within-nest shares from the dataset.
    Returns:
        dict: {market_id: gamma vector for that market}
    """
    gamma_dict = {}
    for market_id in df[market_col].unique():
        market_df = df[df[market_col] == market_id].copy()
        # Use existing within-nest shares instead of recalculating
        within_nest_shares = market_df[within_share_col].values
        if isinstance(sigma, dict):
            sigma_g = market_df[nest_col].map(sigma)
        else:
            sigma_g = sigma
        term = (1 / (1 - sigma_g)) * (1 - sigma_g * within_nest_shares - (1 - sigma_g) * market_df[share_col])
        gamma = beta_N * market_df[share_col] * term
        gamma_dict[market_id] = gamma.values
    return gamma_dict

In [130]:
# Elasticity with network effects
def compute_total_derivatives(df, eta_dict, gamma_dict, v_2, is_ev_col='is_electric', market_col='market_ids'):
    """
    Compute total derivatives of market shares w.r.t. prices (with network effects).
    Args:
        df (pd.DataFrame): DataFrame with all markets.
        eta_dict (dict): {market_id: JxJ eta matrix for that market}.
        gamma_dict (dict): {market_id: gamma vector for that market}.
        v_2 (float): Sensitivity of charging stations to EV sales.
        is_ev_col (str): Column name for EV indicator.
        market_col (str): Column name for market IDs.
    Returns:
        dict: {market_id: JxJ matrix of total derivatives for each market}.
    """
    total_derivatives_dict = {}
    for market_id in df[market_col].unique():
        market_df = df[df[market_col] == market_id].reset_index(drop=True)
        eta = eta_dict[market_id]
        gamma = gamma_dict[market_id]
        is_ev = market_df[is_ev_col].values.astype(bool)
        J = len(gamma)
        total_derivatives = np.zeros((J, J))
        # Use pre-computed EV_share instead of calculating from individual shares
        s_ev = market_df['EV_share'].iloc[0]  # EV_share is the same for all products in a market
        sum_gamma_ev = np.sum(gamma[is_ev])
        eta = np.asarray(eta)  
        for j in range(J):
            for k in range(J):
                if is_ev[j]:
                    feedback_term = 0
                    denom = s_ev - v_2 * sum_gamma_ev
                    if denom != 0:
                        feedback_term = v_2 * gamma[j] * np.sum(eta[is_ev, k]) / denom
                    total_derivatives[j, k] = eta[j, k] + feedback_term
                else:
                    total_derivatives[j, k] = eta[j, k]
        total_derivatives_dict[market_id] = total_derivatives
    return total_derivatives_dict

In [132]:
# Calculate gamma_dict for all markets
beta_N = demand_param.params['log_charging_stock_hat'] + demand_param.params['is_electric:log_charging_stock_hat']
sigma = demand_param.params['log_sj_g']

gamma_dict = compute_gamma_dict(
    beta_N=beta_N,
    df=df,
    nest_col='nesting_ids',
    market_col='market_ids',
    share_col='shares',
    within_share_col='within_nest_shares',
    sigma=sigma
)

# Compute total derivatives using derivative_matrices as eta_dict and gamma_dict as input
total_derivatives_dict = compute_total_derivatives(
    df=df,
    eta_dict=derivative_matrices,
    gamma_dict=gamma_dict,
    v_2=charging_param.params['log(EV_stock)'], 
    is_ev_col='is_electric',
    market_col='market_ids'
)

In [ ]:
# Convert derivatives to elasticities - OPTIMIZED VERSION
def derivatives_to_elasticities_fast(derivative_dict, df, price_col='net_prices', market_col='market_ids', model_col='model'):
    """
    Convert derivative matrices to elasticity matrices using vectorized operations.
    Elasticity[j,k] = Derivative[j,k] * (price_k / share_j)
    """
    elasticity_dict = {}
    
    for market_id, derivative_matrix in derivative_dict.items():
        market_df = df[df[market_col] == market_id].copy()
        prices = market_df[price_col].values
        shares = market_df['shares'].values
        
        # Create price and share matrices for vectorized computation
        price_matrix = np.tile(prices, (len(prices), 1))  # Each row has all prices
        share_matrix = np.tile(shares, (len(shares), 1)).T  # Each column has all shares
        
        # Avoid division by zero
        share_matrix = np.where(share_matrix == 0, np.nan, share_matrix)
        
        # Vectorized elasticity calculation
        elasticity_values = derivative_matrix.values * price_matrix / share_matrix
        
        # Replace NaN with 0
        elasticity_values = np.nan_to_num(elasticity_values, nan=0.0)
        
        # Create DataFrame with same index and columns
        elasticity_matrix = pd.DataFrame(
            elasticity_values, 
            index=derivative_matrix.index, 
            columns=derivative_matrix.columns
        )
        
        elasticity_dict[market_id] = elasticity_matrix
    
    return elasticity_dict

print("🚀 Computing elasticities from derivatives (OPTIMIZED)...")
# Compute elasticities from derivatives (without network effects)
elasticity_matrices = derivatives_to_elasticities_fast(
    derivative_dict=derivative_matrices,
    df=df,
    price_col='net_prices',
    market_col='market_ids',
    model_col='model'
)

print("🚀 Computing elasticities from total derivatives (OPTIMIZED)...")
# Compute elasticities from total derivatives (with network effects) - OPTIMIZED
total_elasticity_matrices = {}
for market_id in total_derivatives_dict.keys():
    market_df = df[df['market_ids'] == market_id].copy()
    prices = market_df['net_prices'].values
    shares = market_df['shares'].values
    model_ids = market_df['model'].values
    
    # Convert total derivatives to elasticities using vectorized operations
    total_deriv_matrix = total_derivatives_dict[market_id]
    
    # Create price and share matrices
    price_matrix = np.tile(prices, (len(prices), 1))
    share_matrix = np.tile(shares, (len(shares), 1)).T
    
    # Avoid division by zero
    share_matrix = np.where(share_matrix == 0, np.nan, share_matrix)
    
    # Vectorized computation
    total_elasticity_values = total_deriv_matrix * price_matrix / share_matrix
    total_elasticity_values = np.nan_to_num(total_elasticity_values, nan=0.0)
    
    # Convert to DataFrame with proper index/columns
    total_elasticity_df = pd.DataFrame(
        total_elasticity_values, 
        index=model_ids, 
        columns=model_ids
    )
    total_elasticity_matrices[market_id] = total_elasticity_df

print("✅ COMPLETED elasticity computation:")
print(f"   - Without network effects: {len(elasticity_matrices)} markets")
print(f"   - With network effects: {len(total_elasticity_matrices)} markets")

# Quick performance check
sample_market = list(elasticity_matrices.keys())[0]
print(f"\n📊 Sample results for market {sample_market}:")
print(f"   - Matrix size: {elasticity_matrices[sample_market].shape}")
print(f"   - Own-price elasticities range: [{np.diag(elasticity_matrices[sample_market]).min():.2f}, {np.diag(elasticity_matrices[sample_market]).max():.2f}]")


🚀 Computing elasticities from derivatives (OPTIMIZED)...
🚀 Computing elasticities from total derivatives (OPTIMIZED)...
✅ COMPLETED elasticity computation:
   - Without network effects: 155 markets
   - With network effects: 155 markets

📊 Sample results for market P01Y2019:
   - Matrix size: (140, 140)
   - Own-price elasticities range: [-31.36, -1.90]


In [134]:
# ===============================================================================
# EXPORT ELASTICITY RESULTS FOR POST-ESTIMATION ANALYSIS (NOTEBOOK 8)
# ===============================================================================

print("💾 EXPORTING ELASTICITY RESULTS FOR POST-ESTIMATION ANALYSIS")
print("="*65)

# 1. Export elasticity matrices (without network effects)
print("📊 Exporting elasticity matrices (without network effects)...")
with open('elasticity_matrices.pkl', 'wb') as f:
    pickle.dump(elasticity_matrices, f)

# 2. Export total elasticity matrices (with network effects)
print("📊 Exporting total elasticity matrices (with network effects)...")
with open('total_elasticity_matrices.pkl', 'wb') as f:
    pickle.dump(total_elasticity_matrices, f)

# 3. Export gamma dictionary (charging station semi-elasticities)
print("📊 Exporting gamma dictionary...")
with open('gamma_dict.pkl', 'wb') as f:
    pickle.dump(gamma_dict, f)

print("\n✅ All elasticity results exported successfully!")
print("Files created for notebook 8:")
print("   • elasticity_matrices.pkl - Price elasticities WITHOUT network effects")
print("   • total_elasticity_matrices.pkl - Price elasticities WITH network effects")  
print("   • gamma_dict.pkl - Charging station semi-elasticities")
print("\n🎯 Ready for post-estimation analysis in notebook 8!")

💾 EXPORTING ELASTICITY RESULTS FOR POST-ESTIMATION ANALYSIS
📊 Exporting elasticity matrices (without network effects)...
📊 Exporting total elasticity matrices (with network effects)...
📊 Exporting gamma dictionary...

✅ All elasticity results exported successfully!
Files created for notebook 8:
   • elasticity_matrices.pkl - Price elasticities WITHOUT network effects
   • total_elasticity_matrices.pkl - Price elasticities WITH network effects
   • gamma_dict.pkl - Charging station semi-elasticities

🎯 Ready for post-estimation analysis in notebook 8!
